In [6]:
import numpy as np
import matplotlib.pyplot as plt
from numba import jit
from scipy import interpolate
from scipy.optimize import minimize
from itertools import product

In [7]:
@jit(nopython=True)
def quad_P(mu_c, mu, alpha, beta, delP, P_c):
    t1 = alpha*(mu - mu_c)**2
    t2 = beta*(mu - mu_c)
    t3 = (1+delP)*P_c
    re = t1 + t2 + t3
    return re

@jit(nopython=True)
def alpha(delP, P_q, P_h, P_c, mu_c, mu_c_h, mu_c_q):
    if (mu_c_q - mu_c_h) == 0 or (mu_c_h - mu_c) == 0 or (mu_c_q - mu_c) == 0:
        re = -1
    else:
        t1 = (P_q - (1+delP)*P_c)/(mu_c_q - mu_c)
        t2 = (P_h - (1+delP)*P_c)/(mu_c_h - mu_c)
        re = (t1 - t2)/(mu_c_q - mu_c_h)
    return re

@jit(nopython=True)
def beta(delP, P_q, P_h, P_c, mu_c, mu_c_h, mu_c_q):
    if (mu_c_q - mu_c_h) == 0 or (mu_c_h - mu_c) == 0 or (mu_c_q - mu_c) == 0:
        re = -1
    else:
        t1 = (P_q - (1+delP)*P_c)/(mu_c_q - mu_c)
        t2 = (P_h - (1+delP)*P_c)/(mu_c_h - mu_c)
        t3 = (P_q - P_h)/(mu_c_q - mu_c_h)
        re = t1 + t2 - t3
    return re

def find_first_larger_index(lst, value):
    return next((i for i, x in enumerate(lst) if x > value), None)

def gradient(n, m):
    
    if len(n) != len(m):
        raise ValueError("n and m must have the same length.")
    
    gradients = []
    for i in range(len(n) - 1):  # Loop up to the second-to-last element
        grad = (n[i + 1] - n[i]) / (m[i + 1] - m[i])
        gradients.append(grad)
    
    return gradients

def compute_n_e(P, mu):
    """Compute number density (n_quad) and energy density (e_quad) from pressure (P) and chemical potential (mu)."""
    n_quad = gradient(P, mu)
    e_quad = (n_quad * mu[:-1]) - P[:-1]
    cs_quad = gradient(P[:-1],e_quad)
    return n_quad, e_quad, cs_quad

def find_valid_parameters(delta_P_arr, y_intersect, x_intersect, p_total, mu_total):
    A, B = [], []
    mu_c_h_arr, mu_c_q_arr = [], []
    P_c_h_arr, P_c_q_arr = [], []
    delta_P_found = []

    index = find_first_larger_index(p_total, y_intersect)
    #print(index)
    x1 = np.array(mu_total[:index])
    y1 = np.array(p_total[:index])

    x2 = np.array(mu_total[index:])
    y2 = np.array(p_total[index:])

    P_h_int_func = interpolate.interp1d(x1, y1, fill_value="extrapolate")
    P_q_int_func = interpolate.interp1d(x2, y2, fill_value="extrapolate")

    mu_new = np.linspace(1050, 1385, 1000)
    P_had_array = P_h_int_func(mu_new)
    P_quark_array = P_q_int_func(mu_new)
    #plt.plot(x1,y1)
    #plt.plot(x2,y2)
    #plt.show()
    #plt.plot(mu_new,P_had_array)
    #plt.plot(mu_new,P_quark_array)
    #plt.show()
    y_diff = np.abs(P_had_array - P_quark_array)

    for delta_P in delta_P_arr:
        found = False

        # Iterate through all combinations of mu_c_h and mu_c_q
        for mu_c_h, mu_c_q in product(mu_new, repeat=2):
            if found:
                break

            P_h = P_h_int_func(mu_c_h)
            P_q = P_q_int_func(mu_c_q)

            alp = alpha(delta_P, P_q, P_h, y_intersect, x_intersect, mu_c_h, mu_c_q)
            bet = beta(delta_P, P_q, P_h, y_intersect, x_intersect, mu_c_h, mu_c_q)

            # Check if alpha is positive
            if alp <= 0:
                continue

            # Calculate the mixed phase pressure
            P_m = quad_P(x_intersect, mu_new, alp, bet, delta_P, y_intersect)

            # Split the pressure difference calculation
            min_idx = np.argmin(y_diff)
            m_h_P_diff = P_m[:min_idx] - P_had_array[:min_idx]
            m_q_P_diff = P_m[min_idx:] - P_quark_array[min_idx:]

            # Validate the pressure conditions
            if np.all(m_h_P_diff >= 0) and np.all(m_q_P_diff >= 0):
                #print(np.min(m_h_P_diff), np.min(m_q_P_diff))
                delta_P_found.append(delta_P)
                A.append(alp)
                B.append(bet)
                mu_c_h_arr.append(mu_c_h)
                mu_c_q_arr.append(mu_c_q)
                P_c_h_arr.append(P_h)
                P_c_q_arr.append(P_q)
                found = True

    print(f"Found {len(A)} valid sets of parameters.")
    return A, B, mu_c_h_arr, mu_c_q_arr, P_c_h_arr, P_c_q_arr, delta_P_found

In [ ]:
def mixed_phase(mu_total, p_total,n_total, A, B, mu_c_h_arr, mu_c_q_arr, P_c_h_arr, P_c_q_arr, mu_crit, P_crit, delta_P_arr):
    P_arr_dp = []
    E_arr_dp = []
    number_density_dp = []
    mu_arr_dp = []

    for i in range(len(A)):
        index_1 = find_first_larger_index(mu_total, mu_c_h_arr[i])
        index_2 = find_first_larger_index(mu_total, mu_c_q_arr[i])

        mu_mixed = np.linspace(mu_total[index_1], mu_total[index_2], len(mu_total[index_1:index_2]))

        P_m = quad_P(mu_crit, mu_mixed, A[i], B[i], delta_P_arr[i], P_crit)
        P_had_half = p_total[:index_1]
        P_quark_half = p_total[index_2:]

        grad_had = n_total[:index_1]
        grad_m = gradient(P_m,mu_mixed)
        grad_quark = n_total[index_2:]

        mu_arr = np.concatenate((mu_total[:index_1], mu_mixed[:-1], mu_total[index_2:]))

        grad_dummy = np.concatenate((grad_had,grad_m,grad_quark))
        P_arr_dummy = np.concatenate((P_had_half,P_m[:-1],P_quark_half))

        e_dummy = (grad_dummy*mu_arr)-P_arr_dummy

        P_arr_dp.append(P_arr_dummy)
        E_arr_dp.append(e_dummy)
        number_density_dp.append(grad_dummy)
        mu_arr_dp.append(mu_arr)

    return P_arr_dp, E_arr_dp, number_density_dp, mu_arr_dp

In [ ]:
def smoothness_eos(pressure, mu, number_density, p_crit, mu_crit):
    delta_P_arr = [0.03, 0.0325, 0.035, 0.0375, 0.04, 0.0425, 0.045, 0.0475, 0.05]
    delta_P_arr = [0.02, 0.06, 0.09, 0.13, 0.15, 0.17]
    
    A, B, mu_c_h_arr, mu_c_q_arr, P_c_h_arr, P_c_q_arr, delta_P_found = find_valid_parameters(delta_P_arr, p_crit, mu_crit, pressure, mu)
    P_arr_dp, E_arr_dp, number_density_dp, mu_arr_dp = mixed_phase(mu, pressure, number_density, A, B, mu_c_h_arr, mu_c_q_arr, P_c_h_arr, P_c_q_arr, mu_crit[40], p_crit, delta_P_arr)
    
    return P_arr_dp, E_arr_dp, number_density_dp, mu_arr_dp

In [10]:
def max_smoothness_eos(pressure, mu, p_crit, mu_crit):
    a =[]
    b =[]
    P_opt=[]

    P_crit_1 = p_crit*(1+0.01)
    P_crit_2 = p_crit*(1+0.2)

    P_crit_11 = np.linspace(P_crit_1,P_crit_2,1000)

    for P in P_crit_11:
        # Define your points
        x1, y1 = mu[0], pressure[0]
        x2, y2 = mu_crit, P

        # For inequality constraint defining hadronic part.
        x_values = np.array(mu[1:])  # Hadronic Part x_values
        f_values = np.array(pressure[1:])    # Corresponding y_values

        # Define the quadratic function
        def quadratic(x, a, b):
            return a * (x - x2)**2 + b * (x - x2) + y2

        # Objective function: sum of squared differences
        def objective(params):
            a, b = params
            return 0

        # Initial guess for parameters
        initial_guess = [0, 0]

        # Constraints to pass through (x1, y1)
        def eq_constraint(params):
            a, b = params
            return quadratic(x1, a, b) - y1

        def ineq_constraint(params):
            a, b = params
            return quadratic(x_values, a, b) - f_values
        

        # Define the constraint in the form required by 'minimize'
        constraints = [
            {'type': 'eq', 'fun': eq_constraint},
            {'type': 'ineq', 'fun': ineq_constraint}
        ]
        # Perform the optimization
        result = minimize(objective, initial_guess, constraints=constraints)
        
        if result.success:
            a_opt, b_opt = result.x
            a.append(a_opt)
            b.append(b_opt)
            P_opt.append(P)
            #print(f"For P_crit = {P}: Optimal parameters found: a = {a_opt}, b = {b_opt}")

        return a, b, P_opt[0]